## Investigating CHIME FD spectra variability in DRAGONS high complexity regions
## A Ordog, Feb-Mar 2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import healpy as hp
from reproject import reproject_from_healpix, reproject_to_healpix
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm

from RMtools_1D import do_RMsynth_1D
from RMtools_1D import do_RMclean_1D

In [ ]:
def chime_hpx_hdr(hdr_hpx_in,hdr_car_chime,nside):

    hpx_hdr = hdr_hpx_in.copy()
    hpx_hdr['NAXIS1'] = hp.nside2npix(nside)
    hpx_hdr['NAXIS2'] = hdr_car_chime['NAXIS3']
    hpx_hdr['NSIDE']  = nside
    hpx_hdr['CRPIX2'] = hdr_car_chime['CRPIX3']
    hpx_hdr['CDELT2'] = hdr_car_chime['CDELT3']
    hpx_hdr['CRVAL2'] = hdr_car_chime['CRVAL3']
    hpx_hdr['BUNIT']  = 'Jy/beam/RMSF'
    del hpx_hdr['DATE']
    del hpx_hdr['INSTRUME']
    
    return hpx_hdr

## Read in DRAGONS Faraday depth cube

In [ ]:
dragons_hdu    = fits.open('/srv/data/dragons/dragons-canfar-Nov2025/faraday_synthesis/dragons_FDF_clean_tot_Kgal.hpx.fits')
dragons_FD_hpx = dragons_hdu[0].data
dragons_hdr    = dragons_hdu[0].header


## Read in best guess of latest CHIME FD map and convert to Healpix if doesn't exist yet

In [ ]:
chime_hpx_made = True

In [ ]:
if chime_hpx_made:
    chime_hdu    = fits.open('/srv/data/chime/chime_FD_May2024_400_729/FDF_clean_tot.hpx512.fits')
    chime_FD_hpx = chime_hdu[0].data
    chime_hdr    = chime_hdu[0].header
else:
    nside = 512
    chime_hdu = fits.open('/srv/data/chime/chime_FD_May2024_400_729/FDF_clean_tot.fits')
    chime_cube = chime_hdu[0].data
    chime_hdr  = chime_hdu[0].header
    print(chime_cube.shape)
    
    chime_hdr_2D = WCS(chime_hdr).dropaxis(2).to_header()
    print(repr(chime_hdr_2D))

    chime_FD_hpx = np.empty([chime_cube.shape[0],hp.nside2npix(nside)])

    for i in range(0,chime_cube_hpx.shape[0]):
        print(i)
        chime_FD_hpx[i], footprint = reproject_to_healpix((chime_cube[i], chime_hdr_2D),
                                                            coord_system_out='G',nside=nside)
    chime_FD_hpx = chime_hpx_hdr(dragons_hdr,chime_hdr,nside)
    fits.writeto('/srv/data/chime/chime_FD_May2024_400_729/FDF_clean_tot.hpx512.fits', 
                 chime_cube_hpx, chime_hdr_hpx, overwrite=False)

## Get CHIME and DRAGONS nside

In [ ]:
nside_dragons = dragons_hdr['NSIDE']
nside_chime   = chime_hdr['NSIDE']

print(nside_dragons)
print(nside_chime)

## Read in convolved CHIME Q and U data

In [ ]:
chime_Q_hdu = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/Q_400_729_Mar2026_gal_conv.fits')
chime_U_hdu = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2026_gal_conv.fits')

chime_Q_hpx = hp.ud = chime_Q_hdu[0].data
chime_U_hpx = chime_U_hdu[0].data

freq_chime = WCS(chime_Q_hdu[0].header).all_pix2world(0,range(chime_Q_hpx.shape[0]),0)[1]
print(len(freq_chime))

print(chime_Q_hpx.shape)
print(chime_U_hpx.shape)


## Downsample CHIME FD cube to nside=256, DRAGONS FD cube and CHIME Q, U cubes to nside=64

In [ ]:
######################
nside_coarse = 64
nside_chime_new = 256
######################

chime_FD_hpx = hp.ud_grade(chime_FD_hpx, nside_chime_new)
print(chime_FD_hpx.shape)

chime_Q_hpx = hp.ud_grade(chime_Q_hpx, nside_coarse)
print(chime_Q_hpx.shape)

chime_U_hpx = hp.ud_grade(chime_U_hpx, nside_coarse)
print(chime_U_hpx.shape)

dragons_FD_hpx = hp.ud_grade(dragons_FD_hpx, nside_coarse)
print(dragons_FD_hpx.shape)


## Read in DRAGONS Faraday complexity map and downsample to nside=64

In [ ]:
hdu_compl = fits.open('/srv/data/dragons/dragons-canfar-Nov2025/faraday_synthesis/dragons_Npeaks_8sigma.hpx.fits')
Npeaks_dragons = hdu_compl[0].data
print(Npeaks_dragons.shape)

Npeaks_dragons = hp.ud_grade(Npeaks_dragons, nside_coarse)
print(Npeaks_dragons.shape)

## Match the pixels between CHIME and coarse resolution (nside=64)

In [ ]:
def get_matched_pixels(lon, lat, nside_fine, nside_coarse):

    idx_coarse = hp.ang2pix(nside_coarse, lon, lat, lonlat=True)

    ratio = (nside_fine // nside_coarse) ** 2  # number of fine pixels per coarse pixel

    ipix_coarse_nested = hp.ring2nest(nside_coarse, idx_coarse)

    # Get fine children (NESTED), then convert back to RING
    fine_nested = np.arange(ipix_coarse_nested * ratio, (ipix_coarse_nested + 1) * ratio)
    fine_ring   = hp.nest2ring(nside_fine, fine_nested)

    #test = hp.pix2ang(nside_coarse, idx_coarse, lonlat=True)
    lon_fine,   lat_fine   = hp.pix2ang(nside_fine, fine_ring, lonlat=True)  
    lon_coarse, lat_coarse = hp.pix2ang(nside_coarse, idx_coarse, lonlat=True)  

    fig,ax = plt.subplots(1,1,figsize=(4,4))
    ax.scatter(lon_coarse, lat_coarse)
    ax.scatter(lon_fine, lat_fine)
    ax.set_aspect('equal')

    return fine_ring, idx_coarse, lon_fine, lat_fine, lon_coarse, lat_coarse

In [ ]:
#idx_list = np.where(Npeaks_dragons == 2)[0]
#for idx in idx_list:
#    print(hp.pix2ang(nside,idx,lonlat=True))

In [ ]:
lon_list = [60, 170,90, 90,130,125,120,90,137]
lat_list = [40,-35,-20,-30, 45, 45, 45,35, 7]
################
lon_t = lon_list[1]
lat_t = lat_list[1]
################

idx_chime, idx_coarse, lon_chime, lat_chime, lon_coarse, lat_coarse = get_matched_pixels(lon_t, lat_t, nside_chime_new, nside_coarse)

str_lon = str(np.round(lon_coarse,1))
str_lat = str(np.round(lat_coarse,1))

hp.mollview(chime_FD_hpx[400],min=0,max=1, title=r'$\ell=$'+str_lon+', $b=$'+str_lat)
hp.projscatter(lon_chime, lat_chime, lonlat=True, marker='o', color='white', s=1)
hp.projscatter(lon_coarse, lat_coarse, lonlat=True, marker='o', color='red', s=1)

hp.mollview(chime_Q_hpx[1],min=-1,max=1, title=r'$\ell=$'+str_lon+', $b=$'+str_lat)
hp.projscatter(lon_chime, lat_chime, lonlat=True, marker='o', color='white', s=1)
hp.projscatter(lon_coarse, lat_coarse, lonlat=True, marker='o', color='red', s=1)

#plt.savefig('/home/aordog/Dropbox/plots/gmims/chime-dragons/spectra_Feb2026/map_'+str_lon+'_'+str_lat+'.png')

## RM synthesis on convolved CHIME data

In [ ]:
dq = np.array([0.01]*len(freq_chime))
du = np.array([0.01]*len(freq_chime))

chime_Q_hpx_LOS = chime_Q_hpx[:,idx_coarse]
chime_U_hpx_LOS = chime_U_hpx[:,idx_coarse]

idx_bad = np.where(abs(chime_Q_hpx_LOS) > 200)
chime_Q_hpx_LOS[idx_bad] = np.nan

idx_bad = np.where(abs(chime_U_hpx_LOS) > 200)
chime_U_hpx_LOS[idx_bad] = np.nan

mDict, aDict = do_RMsynth_1D.run_rmsynth([freq_chime, 
                                          chime_Q_hpx_LOS, 
                                          chime_U_hpx_LOS, 
                                          dq, du],
                                          phiMax_radm2=200,
                                          dPhi_radm2=0.5)

mDict_cl, aDict_cl = do_RMclean_1D.run_rmclean(mDict, aDict, 0.4)


In [ ]:
FD_ax =  np.arange(-200, 200.5, 0.5) # same FD gridding for both
cmap = cm.get_cmap('viridis') 
colors = cmap(np.linspace(0, 1, len(idx_chime)))
fs = 16

fig, ax = plt.subplots(1,1, figsize=(16,6))
ax2 = ax.twinx()
#for i in range(0,len(idx_chime)):
#    ax2.plot(FD_ax, chime_FD_hpx[:,idx_chime[i]], color=colors[i], linewidth=1)

ax.plot(FD_ax, dragons_FD_hpx[:,idx_coarse], color='k', linewidth=3)
ax2.plot(aDict_cl['phiArr_radm2'], abs(aDict_cl['cleanFDF']), color='blue', linewidth=3)
ax2.plot(FD_ax, np.nanmean(chime_FD_hpx[:,idx_chime],axis=1), color='red', linewidth=3)

ax.set_xlim(-50,50)
ax.set_ylim(0,)
ax.set_xticks(np.arange(-50,55,5))
ax.grid(axis='x')

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)

ax.tick_params(axis='both', labelsize=fs, left=True, right=False, 
                   top=True, which='both', width=2, length=6)

ax2.tick_params(axis='both', labelsize=fs, left=False, right=True, 
                   top=True, which='both', width=2, length=6)

ax.set_zorder(ax2.get_zorder()+1)
ax.patch.set_visible(False)

ax2.set_ylabel('CHIME PI (Jy/beam/RMSF)',fontsize=fs)
ax.set_ylabel('DRAGONS PI (K/RMSF)',fontsize=fs)
ax.set_xlabel(r'Faraday depth (rad/m$^2$)',fontsize=fs)

fig.suptitle(r'$\ell=$'+str_lon+', $b=$'+str_lat, fontsize=fs+4)
plt.savefig('/home/aordog/Dropbox/plots/gmims/chime-dragons/spectra_Mar2026/spectra_'+str_lon+'_'+str_lat+'_2.png')

In [ ]:
fs = 14

dragons_max = np.nanmax(dragons_cube_hpx[:,idx_dragons])
chime_max = np.nanmax(chime_cube_hpx[:,idx_chime])

fig = plt.figure(figsize=(20,12))

plt.subplots_adjust(left=0.05, 
                    bottom=0.08, 
                    right=0.95, 
                    top=0.95, 
                    hspace=0.1,
                    wspace=0.1)

gs = gridspec.GridSpec(ncols=4, nrows=4, figure=fig)
axs = []
for i in range(0,4):
    for j in range(0,4):
        axs.append(fig.add_subplot(gs[j,i]))

for i in range(0,16):

    axs[i].plot(FD_ax, dragons_cube_hpx[:,idx_dragons], color='C0')
    ax2 = axs[i].twinx()
    ax2.plot(FD_ax, chime_cube_hpx[:,idx_chime[i]], color='k')
    axs[i].set_xlim(-80,80)
    axs[i].set_ylim(0,dragons_max)
    ax2.set_ylim(0,chime_max)
    axs[i].grid(True)
    
    for spine in axs[i].spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)


    axs[i].tick_params(axis='both', labelsize=fs, left=True, right=False, 
                       top=True, which='both', width=2, length=6)

    ax2.tick_params(axis='both', labelsize=fs, left=False, right=True, 
                       top=True, which='both', width=2, length=6)

    if i in range(0,12):
        ax2.set_yticklabels([])
    else:
        ax2.set_ylabel('CHIME PI (Jy/beam/RMSF)',fontsize=fs)
    if i in range(4,16):
        axs[i].set_yticklabels([])
    else:
        axs[i].set_ylabel('DRAGONS PI (K/RMSF)',fontsize=fs)
    if i in [0,1,2,4,5,6,8,9,10,12,13,14]:
        axs[i].set_xticklabels([])
    else:
        axs[i].set_xlabel(r'Faraday depth (rad/m$^2$)',fontsize=fs)

str_lon = str(np.round(lon_dragons,1))
str_lat = str(np.round(lat_dragons,1))

fig.suptitle(r'$\ell=$'+str_lon+', $b=$'+str_lat, fontsize=fs+6)

#plt.savefig('/home/aordog/Dropbox/plots/gmims/chime-dragons/spectra_Feb2026/spectra_'+str_lon+'_'+str_lat+'.png')